# 1. Đọc dữ liệu gốc

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Tìm thư mục gốc IoT_final
ROOT = None

current = Path.cwd().resolve()

for folder in [current, *current.parents]:
    if (folder / "project").is_dir() and (folder / "result").is_dir():
        ROOT = folder
        break

if ROOT is None:
    raise FileNotFoundError("Không tìm thấy thư mục IoT_final")

# Dataset gốc dùng chung của nhóm
DATA_PATH = (
    ROOT / "result" / "preprocessing"
    / "raw data" / "RT_IOT2022.csv"
)

# Folder kết quả Cleaning của bạn
RESULT_DIR = (
    ROOT / "result" / "preprocessing"
    / "C01_Data_Cleaning_ThanhDai"
)

TABLE_DIR = RESULT_DIR / "tables"

OUTPUT_DIR = (
    RESULT_DIR / "data" / "processed"
)

TABLE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Đọc dataset
df = pd.read_csv(DATA_PATH)

print("Thư mục dự án:", ROOT)
print("Dataset:", DATA_PATH)
print("Kích thước ban đầu:", df.shape)

display(df.head())

Thư mục dự án: D:\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project
Dataset: D:\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\result\preprocessing\raw data\RT_IOT2022.csv
Kích thước ban đầu: (123117, 85)


,Unnamed: 0,id.orig_p,id.resp_p,proto,service,flow_duration,fwd_pkts_tot,bwd_pkts_tot,fwd_data_pkts_tot,bwd_data_pkts_tot,...,active.std,idle.min,idle.max,idle.tot,idle.avg,idle.std,fwd_init_window_size,bwd_init_window_size,fwd_last_window_size,Attack_type
0,0,38667,1883,tcp,mqtt,32.011598,9,5,3,3,...,0.0,2.972918e+07,2.972918e+07,2.972918e+07,2.972918e+07,0.0,64240,26847,502,MQTT_Publish
1,1,51143,1883,tcp,mqtt,31.883584,9,5,3,3,...,0.0,2.985528e+07,2.985528e+07,2.985528e+07,2.985528e+07,0.0,64240,26847,502,MQTT_Publish
2,2,44761,1883,tcp,mqtt,32.124053,9,5,3,3,...,0.0,2.984215e+07,2.984215e+07,2.984215e+07,2.984215e+07,0.0,64240,26847,502,MQTT_Publish
3,3,60893,1883,tcp,mqtt,31.961063,9,5,3,3,...,0.0,2.991377e+07,2.991377e+07,2.991377e+07,2.991377e+07,0.0,64240,26847,502,MQTT_Publish
4,4,51087,1883,tcp,mqtt,31.902362,9,5,3,3,...,0.0,2.981470e+07,2.981470e+07,2.981470e+07,2.981470e+07,0.0,64240,26847,502,MQTT_Publish


### Nhận xét: Đọc dữ liệu gốc

- Bộ dữ liệu RT_IOT2022 có **123.117 bản ghi và 85 thuộc tính**.
- Dữ liệu chứa các đặc trưng của lưu lượng mạng IoT và nhãn mục tiêu `Attack_type`.
- File dữ liệu gốc được đọc để thực hiện làm sạch và không bị chỉnh sửa trực tiếp.

# 2. Kiểm tra dữ liệu trước khi cleaning

In [2]:
target = "Attack_type"

# Kiểm tra giá trị thiếu
missing_count = int(df.isna().sum().sum())

# Kiểm tra giá trị vô hạn
numeric_df = df.select_dtypes(include=np.number)

inf_count = int(
    np.isinf(numeric_df.to_numpy()).sum()
)

# Kiểm tra cột hằng
constant_cols = df.columns[
    df.nunique(dropna=False) == 1
].tolist()

# Kiểm tra trùng lặp (không tính cột chỉ số)
duplicate_cols = df.columns.drop(
    "Unnamed: 0",
    errors="ignore"
)

duplicate_count = int(
    df.duplicated(subset=duplicate_cols).sum()
)

# Phân bố nhãn trước khi xử lý
class_before = df[target].value_counts()

print("=== CHẤT LƯỢNG DỮ LIỆU BAN ĐẦU ===")

print("Missing Values:", missing_count)
print("Infinity Values:", inf_count)
print("Duplicate Rows:", duplicate_count)
print("Constant Columns:", constant_cols)

print("\n=== PHÂN BỐ NHÃN BAN ĐẦU ===")
display(class_before.to_frame("Count"))

=== CHẤT LƯỢNG DỮ LIỆU BAN ĐẦU ===
Missing Values: 0
Infinity Values: 0
Duplicate Rows: 5195
Constant Columns: ['bwd_URG_flag_count']

=== PHÂN BỐ NHÃN BAN ĐẦU ===


,Count
Attack_type,
DOS_SYN_Hping,94659
Thing_Speak,8108
ARP_poisioning,7750
MQTT_Publish,4146
NMAP_UDP_SCAN,2590
NMAP_XMAS_TREE_SCAN,2010
NMAP_OS_DETECTION,2000
NMAP_TCP_scan,1002
DDOS_Slowloris,534


### Nhận xét: Chất lượng dữ liệu trước Cleaning

**Kết quả kiểm tra:**

- Không phát hiện giá trị thiếu.
- Không phát hiện giá trị vô hạn.
- Có **5.195 dòng trùng lặp** khi không tính cột `Unnamed: 0`.
- Phát hiện một cột hằng là `bwd_URG_flag_count`.
- Bộ dữ liệu gồm 12 lớp thuộc tính `Attack_type`.

**Kết luận:** Cần xử lý các cột không phù hợp và kiểm tra bản ghi trùng lặp.

# 3. Loại bỏ cột chỉ số và cột hằng

In [3]:
columns_to_drop = [
    "Unnamed: 0",
    "bwd_URG_flag_count"
]

# Tạo bản dữ liệu mới, giữ nguyên df gốc
df_work = df.drop(
    columns=columns_to_drop,
    errors="raise"
).copy()

print("=== KẾT QUẢ LOẠI CỘT ===")

print("Kích thước ban đầu:", df.shape)
print("Kích thước sau xử lý:", df_work.shape)
print("Các cột đã loại:", columns_to_drop)

print("\n=== KIỂM TRA CỘT CÒN LẠI ===")
print("Số cột:", len(df_work.columns))
print("Nhãn mục tiêu còn tồn tại:",
      target in df_work.columns)

=== KẾT QUẢ LOẠI CỘT ===
Kích thước ban đầu: (123117, 85)
Kích thước sau xử lý: (123117, 83)
Các cột đã loại: ['Unnamed: 0', 'bwd_URG_flag_count']

=== KIỂM TRA CỘT CÒN LẠI ===
Số cột: 83
Nhãn mục tiêu còn tồn tại: True


### Nhận xét: Loại bỏ cột không phù hợp

**Kết quả thực hiện:**

- Đã loại bỏ cột `Unnamed: 0` vì đây là cột chỉ số.
- Đã loại bỏ `bwd_URG_flag_count` vì thuộc tính này chỉ chứa một giá trị.
- Số lượng thuộc tính giảm từ 85 xuống còn 83.
- Số lượng bản ghi vẫn giữ nguyên là 123.117.

**Kết luận:**

Việc loại bỏ hai cột giúp tránh đưa những thông tin không có giá trị phân biệt vào mô hình.


# 4. Kiểm tra và xử lý nhãn mâu thuẫn

In [4]:
feature_cols = [
    col for col in df_work.columns
    if col != target
]

# Đếm số nhãn khác nhau ứng với mỗi bộ đặc trưng
label_counts = (
    df_work.groupby(
        feature_cols,
        dropna=False,
        sort=False
    )[target]
    .transform("nunique")
)

# Những dòng thuộc nhóm có nhiều hơn một nhãn
conflict_mask = label_counts > 1

df_conflicts = df_work.loc[conflict_mask].copy()

conflict_groups = (
    df_conflicts.groupby(
        feature_cols,
        dropna=False,
        sort=False
    ).ngroups
)

print("=== KIỂM TRA NHÃN MÂU THUẪN ===")

print("Số nhóm mâu thuẫn:", conflict_groups)
print("Số dòng thuộc nhóm mâu thuẫn:", len(df_conflicts))

# Loại các nhóm mâu thuẫn khỏi bản dữ liệu làm việc
df_no_conflicts = df_work.loc[
    ~conflict_mask
].copy()

print("\n=== SAU KHI XỬ LÝ ===")

print("Trước:", df_work.shape)
print("Sau:", df_no_conflicts.shape)

=== KIỂM TRA NHÃN MÂU THUẪN ===
Số nhóm mâu thuẫn: 6
Số dòng thuộc nhóm mâu thuẫn: 128

=== SAU KHI XỬ LÝ ===
Trước: (123117, 83)
Sau: (122989, 83)


### Nhận xét: Xử lý nhãn mâu thuẫn

**Kết quả thực hiện:**

- Phát hiện 6 nhóm bản ghi có đặc trưng đầu vào giống nhau nhưng nhãn mục tiêu khác nhau.
- Tổng cộng 128 bản ghi thuộc các nhóm này.
- Đã loại bỏ toàn bộ 128 bản ghi theo phương án làm sạch được lựa chọn.
- Số lượng bản ghi còn lại là 122.989.

**Kết luận:**

Việc loại bỏ các nhóm mâu thuẫn giúp tạo tập dữ liệu có nhãn nhất quán hơn đối với các đặc trưng hiện có.

# 5. Loại bỏ các bản ghi trùng lặp

In [5]:
rows_before = len(df_no_conflicts)

# Các nhóm mâu thuẫn đã được loại ở Cell 4
df_clean = df_no_conflicts.drop_duplicates(
    keep="first"
).reset_index(drop=True)

removed_duplicates = rows_before - len(df_clean)

print("=== KẾT QUẢ LOẠI TRÙNG ===")

print("Số dòng trước:", rows_before)
print("Số dòng trùng đã loại:", removed_duplicates)
print("Số dòng sau:", len(df_clean))

print("\nKích thước dữ liệu sạch:", df_clean.shape)

print(
    "Số dòng trùng còn lại:",
    df_clean.duplicated().sum()
)

=== KẾT QUẢ LOẠI TRÙNG ===
Số dòng trước: 122989
Số dòng trùng đã loại: 5080
Số dòng sau: 117909

Kích thước dữ liệu sạch: (117909, 83)
Số dòng trùng còn lại: 0


### Nhận xét: Loại bỏ dữ liệu trùng lặp

**Kết quả thực hiện:**

- Sau khi xử lý nhãn mâu thuẫn, dataset còn 122.989 bản ghi.
- Phát hiện và loại bỏ tiếp 5.080 bản ghi trùng hoàn toàn.
- Dataset sau làm sạch còn 117.909 bản ghi.
- Không còn các dòng trùng hoàn toàn trong dữ liệu đã làm sạch.

**Kết luận:**

Việc loại bỏ trùng lặp giúp giảm sự lặp lại không cần thiết trong dữ liệu.

Đồng thời, thao tác này giảm nguy cơ những mẫu trùng hoàn toàn xuất hiện ở cả các tập dữ liệu sau khi chia.

# 6. Kiểm tra dữ liệu sau Cleaning và lưu kết quả

In [6]:
numeric_clean = df_clean.select_dtypes(include=np.number)

# Kiểm tra chất lượng sau Cleaning
assert df_clean.isna().sum().sum() == 0
assert np.isinf(numeric_clean.to_numpy()).sum() == 0
assert df_clean.duplicated().sum() == 0
assert df_clean[target].nunique() == df[target].nunique()

# Kiểm tra nhãn mâu thuẫn còn lại
remaining_conflicts = (
    df_clean.groupby(
        feature_cols,
        dropna=False,
        sort=False
    )[target]
    .transform("nunique")
    .gt(1)
    .sum()
)

assert remaining_conflicts == 0

print("=== KIỂM TRA SAU CLEANING ===")

print("Kích thước ban đầu:", df.shape)
print("Kích thước sau làm sạch:", df_clean.shape)
print("Missing:", df_clean.isna().sum().sum())
print("Duplicate:", df_clean.duplicated().sum())
print("Nhãn mâu thuẫn:", remaining_conflicts)
print("Số lớp:", df_clean[target].nunique())

# So sánh số lượng mẫu từng lớp
comparison = pd.DataFrame({
    "Before": df[target].value_counts(),
    "After": df_clean[target].value_counts()
}).fillna(0).astype(int)

comparison["Removed"] = (
    comparison["Before"] - comparison["After"]
)

print("\n=== PHÂN BỐ NHÃN TRƯỚC VÀ SAU ===")
display(comparison)

# Lưu dataset đã làm sạch
output_path = (
    OUTPUT_DIR / "RT_IOT2022_clean_ThanhDai.csv"
)

df_clean.to_csv(
    output_path,
    index=False
)

# Lưu bảng thống kê
comparison.to_csv(
    TABLE_DIR / "C01_class_distribution_comparison.csv",
    encoding="utf-8-sig"
)

print("\nĐã lưu dữ liệu sạch tại:")
print(output_path)

=== KIỂM TRA SAU CLEANING ===
Kích thước ban đầu: (123117, 85)
Kích thước sau làm sạch: (117909, 83)
Missing: 0
Duplicate: 0
Nhãn mâu thuẫn: 0
Số lớp: 12

=== PHÂN BỐ NHÃN TRƯỚC VÀ SAU ===


,Before,After,Removed
Attack_type,,,
DOS_SYN_Hping,94659,90089,4570
Thing_Speak,8108,7650,458
ARP_poisioning,7750,7620,130
MQTT_Publish,4146,4142,4
NMAP_UDP_SCAN,2590,2582,8
NMAP_XMAS_TREE_SCAN,2010,2010,0
NMAP_OS_DETECTION,2000,2000,0
NMAP_TCP_scan,1002,1002,0
DDOS_Slowloris,534,532,2



Đã lưu dữ liệu sạch tại:
D:\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\result\preprocessing\C01_Data_Cleaning_ThanhDai\data\processed\RT_IOT2022_clean_ThanhDai.csv


### Nhận xét: Kiểm tra kết quả Data Cleaning

**Kết quả sau làm sạch:**

- Dataset giảm từ 123.117 xuống còn 117.909 bản ghi.
- Số cột giảm từ 85 xuống còn 83.
- Không phát hiện giá trị thiếu hoặc vô hạn.
- Không còn bản ghi trùng hoàn toàn.
- Không còn nhóm đặc trưng giống nhau nhưng nhãn mâu thuẫn.
- Vẫn giữ đủ 12 lớp thuộc tính `Attack_type`.

**Kết luận:**

Dataset sau Cleaning đủ điều kiện để tiếp tục thiết kế bước chia dữ liệu.

Chưa thực hiện chuẩn hóa, One-Hot Encoding, Feature Selection hoặc Oversampling.